# Impacto Ambiental de la Expansión Agrícola en Michoacán

Michoacán es el epicentro mundial de la producción de aguacate. A continuación, se presenta un análisis visual estructurado a partir de datos procesados para comprender la relación entre la expansión de este cultivo y la disminución de la masa forestal.


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

base_path = '../data/procesados' if os.path.exists('../data/procesados') else 'data/procesados'

COLOR_BOSQUE = '#2D5A27'
COLOR_EMISIONES = '#B94A41'


## 1. Evolución Histórica de Pérdida Forestal y Emisiones

La siguiente gráfica detalla la pérdida de cobertura forestal anual y las emisiones brutas de CO2 generadas. El gráfico de doble eje permite observar la correlación a lo largo del tiempo.


In [2]:
df_hist = pd.read_csv(f'{base_path}/gfw_historico_perdida_emisiones.csv')

df_hist_agrupado = df_hist.groupby('umd_tree_cover_loss__year').agg({
    'umd_tree_cover_loss__ha': 'sum',
    'gfw_gross_emissions_co2e_all_gases__Mg': 'sum'
}).reset_index()
df_hist_agrupado.columns = ['Anio', 'Hectareas_Perdidas', 'Emisiones_CO2e']

fig = go.Figure()

fig.add_trace(go.Bar(
    x=df_hist_agrupado['Anio'],
    y=df_hist_agrupado['Emisiones_CO2e'],
    name='Emisiones de CO2e',
    marker_color=COLOR_EMISIONES,
    opacity=0.6,
    yaxis='y2'
))

fig.add_trace(go.Scatter(
    x=df_hist_agrupado['Anio'],
    y=df_hist_agrupado['Hectareas_Perdidas'],
    name='Pérdida Forestal (Ha)',
    mode='lines+markers',
    line=dict(color=COLOR_BOSQUE, width=2)
))

fig.update_layout(
    title='Pérdida Forestal vs Emisiones en Michoacán (2001-Presente)',
    xaxis_title='Año',
    yaxis=dict(title='Hectáreas Perdidas', title_font=dict(color=COLOR_BOSQUE)),
    yaxis2=dict(title='Emisiones CO2e (Mg)', title_font=dict(color=COLOR_EMISIONES), overlaying='y', side='right'),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(x=0.01, y=0.99)
)

fig.show()


## 2. Impacto Municipal

El siguiente diagrama de dispersión compara la superficie dedicada al cultivo con la superficie forestal restante por municipio. El tamaño del punto refleja la capacidad productiva.


In [3]:
df_mun = pd.read_csv(f'{base_path}/dataset_maestro_ods15.csv')
df_mun = df_mun.sort_values(by='ha_aguacate', ascending=False)

fig = px.scatter(
    df_mun,
    x='ha_aguacate',
    y='ha_forestal_total',
    size='ton_aguacate',
    color='pct_bosque',
    hover_name='municipio',
    color_continuous_scale='Greens',
    labels={
        'ha_aguacate': 'Superficie de Aguacate (Ha)',
        'ha_forestal_total': 'Superficie Forestal Total (Ha)',
        'pct_bosque': '% de Bosque',
        'ton_aguacate': 'Producción (Ton)'
    },
    title='Superficie Agrícola vs Forestal por Municipio',
    template='plotly_white'
)

fig.update_layout(coloraxis_colorbar=dict(title='% Bosque'))
fig.show()


## 3. Causas de la Pérdida Forestal

El diagrama circular expone los principales impulsores de la deforestación acumulada, permitiendo dimensionar el impacto de la actividad agrícola frente a otros factores.


In [4]:
df_drivers = pd.read_csv(f'{base_path}/gfw_drivers_perdida_completo_es.csv')
df_drivers_agrupado = df_drivers.groupby('drivers_type')['loss_area_ha'].sum().reset_index()

fig = px.pie(
    df_drivers_agrupado,
    names='drivers_type',
    values='loss_area_ha',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel,
    title='Proporción de Pérdida Forestal por Causa',
    labels={'drivers_type': 'Causa', 'loss_area_ha': 'Hectáreas Perdidas'}
)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()


## 4. Calculadora de Impacto Ambiental

Esta sección proyecta las implicaciones ambientales y económicas al autorizar nuevas hectáreas para cultivo, basándose en coeficientes procesados.


In [5]:
import json

def calculadora_impacto(hectareas_nuevas):
    with open(f'{base_path}/impact_coefficients.json', 'r') as f:
        data = json.load(f)
    
    coef = data['calculadora_impacto']['coeficientes']
    rendimiento_kg = hectareas_nuevas * 10000 
    
    agua_litros = rendimiento_kg * coef['agua_litros_por_kg']
    co2_ton = hectareas_nuevas * coef['co2_ton_por_ha_forestal']
    ingresos_usd = (rendimiento_kg / 1000) * coef['precio_estimado_ton_usd']
    empleos = hectareas_nuevas * coef['empleos_por_ha']
    
    impacto_df = pd.DataFrame({
        'Métrica': ['Demanda Hídrica (L)', 'Emisiones CO2e (Ton)', 'Ingreso Estimado (USD)', 'Empleos Directos'],
        'Valor': [agua_litros, co2_ton, ingresos_usd, empleos],
        'Tipo': ['Ambiental', 'Ambiental', 'Económico', 'Socioeconómico']
    })
    
    fig = px.bar(
        impacto_df,
        x='Valor',
        y='Métrica',
        color='Tipo',
        orientation='h',
        text='Valor',
        color_discrete_map={'Ambiental': COLOR_EMISIONES, 'Económico': COLOR_BOSQUE, 'Socioeconómico': '#4E79A7'},
        title=f'Proyección de Impacto para {hectareas_nuevas} Hectáreas Nuevas'
    )
    
    fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
    fig.update_layout(template='plotly_white', xaxis_type='log', xaxis_title='Valor (Escala Logarítmica)')
    fig.show()

calculadora_impacto(15)
